# 00 — Colab setup

Run this first in every new Colab session (select a Colab runtime as the kernel in VS Code).

It will:
1. show the runtime (Python, GPU);
2. mount Google Drive, where data and runs are kept (`MyDrive/dtwin/`);
3. clone or update the code from GitHub and install it;
4. check that the package works by running the tests.

The runtime's local disk is wiped when it disconnects. Anything worth keeping must go under Drive.

In [ ]:
import sys, platform, subprocess
print("Python:", sys.version.split()[0], "|", platform.platform())
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "none (CPU runtime)")

In [ ]:
# Mount Google Drive. If mounting is not supported in your setup, set USE_DRIVE = False:
# outputs then go to /content/dtwin_tmp and are LOST when the runtime disconnects.
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.pop("DTWIN_ROOT", None)
else:
    os.environ["DTWIN_ROOT"] = "/content/dtwin_tmp"
    print("WARNING: not using Drive; results will not persist.")

In [ ]:
# Clone (first time) or update (later sessions) the public repository.
REPO_URL = "https://github.com/plufle/Patient-adaptive-ECG-PPG-hypotension-warning-with-competing-explanation-search.git"
REPO_DIR = "/content/repo"
BRANCH = "main"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Install the package. EXTRAS grows as phases need more (e.g. "dev,sim" in P1, "dev,sim,signal" in P3).
EXTRAS = "dev"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# Make the editable install visible to this already-running kernel.
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)

import dtwin
from dtwin.utils.io import get_paths, on_colab
print("dtwin", dtwin.__version__, "| on_colab:", on_colab())
print(get_paths(create=True))

In [ ]:
# Run the test suite on the Colab runtime.
result = subprocess.run([sys.executable, "-m", "pytest", "-q", os.path.join(REPO_DIR, "tests")],
                        capture_output=True, text=True, cwd=REPO_DIR)
print(result.stdout[-3000:], result.stderr[-2000:])
assert result.returncode == 0, "tests failed"